# Course 10: MLOps with Vertex AI — Manage Features

**Companion notebook for**: `10-mlops-feature-store.html`

## Learning Objectives
- Create a Feature Group backed by BigQuery
- Register features with metadata and descriptions
- Generate training data with point-in-time correct joins (batch serving)
- Fetch latest feature values for online prediction serving
- Configure feature monitoring for drift and staleness detection
- Compare Feature Store-served vs inline feature patterns

## Prerequisites
- GCP project with Vertex AI API and BigQuery API enabled
- Service account or user credentials with Vertex AI and BigQuery permissions
- Python 3.10+

---
## Setup & Dependencies

In [ ]:
%pip install -q google-cloud-aiplatform>=1.38.0 google-cloud-bigquery pandas matplotlib db-dtypes

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# --- GCP Configuration ---
PROJECT_ID = os.environ.get("GCP_PROJECT_ID", "your-project-id")
REGION = os.environ.get("GCP_REGION", "us-central1")
BQ_DATASET = "feature_store_demo"

print(f"Project:    {PROJECT_ID}")
print(f"Region:     {REGION}")
print(f"BQ Dataset: {BQ_DATASET}")

In [ ]:
from google.cloud import aiplatform
from google.cloud import bigquery

# Initialize Vertex AI
aiplatform.init(project=PROJECT_ID, location=REGION)

# Initialize BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID)

print(f"Vertex AI SDK version: {aiplatform.__version__}")
print(f"BigQuery client initialized.")

---
## Section 1: Create Feature Data in BigQuery

The new Vertex AI Feature Store is **BigQuery-backed**. Features are stored as BigQuery tables,
and the Feature Store provides a managed layer for serving, monitoring, and governance.

We will create synthetic feature tables that represent a fraud detection scenario:
- **User features**: aggregated user behavior (account age, transaction history)
- **Transaction features**: per-transaction attributes
- **Labels**: fraud/legitimate labels with timestamps

In [ ]:
# Create BigQuery dataset for feature store demo
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
dataset_ref.location = REGION

try:
    bq_client.create_dataset(dataset_ref, exists_ok=True)
    print(f"Dataset '{BQ_DATASET}' ready.")
except Exception as e:
    print(f"Dataset creation: {e}")

In [ ]:
# Generate synthetic user features with timestamps (for point-in-time demos)
np.random.seed(42)

n_users = 200
n_snapshots = 10  # Feature snapshots per user (one per week)

rows = []
base_date = datetime(2025, 1, 1)

for user_idx in range(n_users):
    user_id = f"user_{user_idx:04d}"
    account_age_base = np.random.randint(30, 1500)
    is_verified = np.random.choice([True, False], p=[0.7, 0.3])
    base_txn_count = np.random.randint(2, 50)
    base_avg_amount = np.random.uniform(20, 2000)

    for week in range(n_snapshots):
        feature_ts = base_date + timedelta(weeks=week)
        # Features evolve slightly over time
        txn_count_7d = max(0, int(base_txn_count + np.random.normal(0, 3)))
        avg_txn_amount_30d = max(0, base_avg_amount + np.random.normal(0, base_avg_amount * 0.1))
        account_age_days = account_age_base + week * 7

        rows.append({
            "user_id": user_id,
            "feature_timestamp": feature_ts.isoformat(),
            "avg_transaction_amount_30d": round(avg_txn_amount_30d, 2),
            "transaction_count_7d": txn_count_7d,
            "account_age_days": account_age_days,
            "is_verified": is_verified,
            "risk_score": round(np.random.uniform(0, 1), 4),
            "device_count": np.random.randint(1, 5),
        })

user_features_df = pd.DataFrame(rows)
print(f"User features DataFrame: {user_features_df.shape}")
print(f"Users: {n_users}, Snapshots per user: {n_snapshots}")
print(f"\nSample:")
user_features_df.head(3)

In [ ]:
# Upload user features to BigQuery
user_features_table = f"{PROJECT_ID}.{BQ_DATASET}.user_features"

job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("user_id", "STRING"),
        bigquery.SchemaField("feature_timestamp", "TIMESTAMP"),
        bigquery.SchemaField("avg_transaction_amount_30d", "FLOAT64"),
        bigquery.SchemaField("transaction_count_7d", "INT64"),
        bigquery.SchemaField("account_age_days", "INT64"),
        bigquery.SchemaField("is_verified", "BOOL"),
        bigquery.SchemaField("risk_score", "FLOAT64"),
        bigquery.SchemaField("device_count", "INT64"),
    ],
)

job = bq_client.load_table_from_dataframe(
    user_features_df, user_features_table, job_config=job_config
)
job.result()  # Wait for completion

table = bq_client.get_table(user_features_table)
print(f"Loaded {table.num_rows} rows to {user_features_table}")

In [ ]:
# Generate labels table with timestamps
# Each label has an entity_id and a timestamp (when the event occurred)
np.random.seed(123)

label_rows = []
for user_idx in range(n_users):
    user_id = f"user_{user_idx:04d}"
    # Each user has 1-3 labeled events at random times
    n_events = np.random.randint(1, 4)
    for _ in range(n_events):
        # Label timestamp: random point within our feature time range
        event_offset_days = np.random.randint(7, n_snapshots * 7)
        label_ts = base_date + timedelta(days=event_offset_days)
        is_fraud = np.random.choice([0, 1], p=[0.95, 0.05])

        label_rows.append({
            "user_id": user_id,
            "label_timestamp": label_ts.isoformat(),
            "is_fraud": is_fraud,
        })

labels_df = pd.DataFrame(label_rows)

# Upload labels to BigQuery
labels_table = f"{PROJECT_ID}.{BQ_DATASET}.fraud_labels"
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
    schema=[
        bigquery.SchemaField("user_id", "STRING"),
        bigquery.SchemaField("label_timestamp", "TIMESTAMP"),
        bigquery.SchemaField("is_fraud", "INT64"),
    ],
)

job = bq_client.load_table_from_dataframe(
    labels_df, labels_table, job_config=job_config
)
job.result()

print(f"Labels: {len(labels_df)} events for {labels_df['user_id'].nunique()} users")
print(f"Fraud rate: {labels_df['is_fraud'].mean():.2%}")
print(f"Uploaded to: {labels_table}")

---
## Section 2: Create Feature Group and Register Features

A **Feature Group** points to a BigQuery table and registers individual columns as features.
This creates a managed layer for discovery, governance, and serving.

In [ ]:
from vertexai.resources.preview import feature_store

# Create a Feature Group backed by the BigQuery table
FEATURE_GROUP_NAME = "user_fraud_features"

try:
    user_fg = feature_store.FeatureGroup.create(
        name=FEATURE_GROUP_NAME,
        source=feature_store.utils.FeatureGroupBigQuerySource(
            uri=f"bq://{user_features_table}",
            entity_id_columns=["user_id"],
        ),
        description="User-level features for fraud detection model",
    )
    print(f"Feature Group created: {user_fg.name}")
    print(f"Resource name: {user_fg.resource_name}")
except Exception as e:
    print(f"Feature Group creation: {e}")
    # If already exists, load it
    user_fg = feature_store.FeatureGroup(FEATURE_GROUP_NAME)
    print(f"Loaded existing Feature Group: {user_fg.name}")

In [ ]:
# Register individual features with metadata
feature_definitions = [
    {
        "name": "avg_transaction_amount_30d",
        "description": "Average transaction amount over the last 30 days (USD)",
    },
    {
        "name": "transaction_count_7d",
        "description": "Number of transactions in the last 7 days",
    },
    {
        "name": "account_age_days",
        "description": "Number of days since account creation",
    },
    {
        "name": "is_verified",
        "description": "Whether user has completed identity verification (KYC)",
    },
    {
        "name": "risk_score",
        "description": "Proprietary risk score (0.0 = low risk, 1.0 = high risk)",
    },
    {
        "name": "device_count",
        "description": "Number of unique devices used by this user",
    },
]

registered_features = []
for feat_def in feature_definitions:
    try:
        feature = user_fg.create_feature(
            name=feat_def["name"],
            description=feat_def["description"],
        )
        registered_features.append(feature)
        print(f"  Registered: {feat_def['name']}")
    except Exception as e:
        print(f"  {feat_def['name']}: {e}")

print(f"\nTotal features registered: {len(registered_features)}")

In [ ]:
# List all features in the feature group (discovery)
features = user_fg.list_features()

print(f"Features in '{FEATURE_GROUP_NAME}':")
print(f"{'Name':<35} {'Description'}")
print("-" * 75)
for f in features:
    print(f"{f.name:<35} {f.description or '(no description)'}")

---
## Section 3: Batch Feature Serving with Point-in-Time Joins

This is the most critical section for the exam. **Point-in-time correctness** means each training
example uses only feature values that were available at the time of the label.

### Why This Matters
Without point-in-time correctness, you get **data leakage**:
- Training metrics look great (AUC 0.98+)
- Production performance is much worse (AUC 0.85)
- The model was "cheating" by seeing future data during training

### How It Works
For each (entity_id, label_timestamp), the Feature Store finds the feature values
that were current **at or before** the label timestamp.

In [ ]:
# Demonstrate point-in-time join using BigQuery SQL
# This is what Feature Store's batch_serve does internally

pit_query = f"""
-- Point-in-time join: get features AS OF each label timestamp
-- This prevents data leakage by only using features available at prediction time

WITH ranked_features AS (
    SELECT
        l.user_id,
        l.label_timestamp,
        l.is_fraud,
        f.avg_transaction_amount_30d,
        f.transaction_count_7d,
        f.account_age_days,
        f.is_verified,
        f.risk_score,
        f.device_count,
        f.feature_timestamp,
        -- Rank features: most recent BEFORE label timestamp = rank 1
        ROW_NUMBER() OVER (
            PARTITION BY l.user_id, l.label_timestamp
            ORDER BY f.feature_timestamp DESC
        ) AS feature_rank
    FROM `{PROJECT_ID}.{BQ_DATASET}.fraud_labels` l
    LEFT JOIN `{PROJECT_ID}.{BQ_DATASET}.user_features` f
        ON l.user_id = f.user_id
        AND f.feature_timestamp <= l.label_timestamp  -- KEY: only past features!
)
SELECT
    user_id,
    label_timestamp,
    is_fraud,
    avg_transaction_amount_30d,
    transaction_count_7d,
    account_age_days,
    is_verified,
    risk_score,
    device_count,
    feature_timestamp AS feature_as_of_timestamp
FROM ranked_features
WHERE feature_rank = 1  -- Take the most recent valid features
ORDER BY user_id, label_timestamp
"""

print("Executing point-in-time join query...")
training_data = bq_client.query(pit_query).to_dataframe()

print(f"\nTraining data generated: {training_data.shape}")
print(f"Fraud rate: {training_data['is_fraud'].mean():.2%}")
print(f"\nSample rows:")
training_data.head(5)

In [ ]:
# Verify point-in-time correctness:
# feature_as_of_timestamp should always be <= label_timestamp

training_data["label_timestamp"] = pd.to_datetime(training_data["label_timestamp"])
training_data["feature_as_of_timestamp"] = pd.to_datetime(training_data["feature_as_of_timestamp"])

time_diff = (training_data["label_timestamp"] - training_data["feature_as_of_timestamp"]).dt.days

print("Point-in-Time Correctness Verification:")
print(f"  All features before labels: {(time_diff >= 0).all()}")
print(f"  Min gap (days): {time_diff.min()}")
print(f"  Max gap (days): {time_diff.max()}")
print(f"  Mean gap (days): {time_diff.mean():.1f}")
print(f"  Median gap (days): {time_diff.median():.1f}")

# Any rows where features are AFTER labels = data leakage!
leaky_rows = (time_diff < 0).sum()
print(f"\n  Data leakage rows: {leaky_rows} (should be 0)")

In [ ]:
# Visualize the point-in-time join for a single user
demo_user = "user_0001"

# Get all feature snapshots for this user
user_snapshots = user_features_df[user_features_df["user_id"] == demo_user].copy()
user_snapshots["feature_timestamp"] = pd.to_datetime(user_snapshots["feature_timestamp"])

# Get all labels for this user
user_labels = labels_df[labels_df["user_id"] == demo_user].copy()
user_labels["label_timestamp"] = pd.to_datetime(user_labels["label_timestamp"])

# Get the training data rows for this user
user_training = training_data[training_data["user_id"] == demo_user]

fig, ax = plt.subplots(figsize=(14, 5))

# Plot feature snapshots
ax.scatter(
    user_snapshots["feature_timestamp"],
    user_snapshots["transaction_count_7d"],
    color="#3b82f6", s=80, zorder=5, label="Feature snapshots",
)
ax.plot(
    user_snapshots["feature_timestamp"],
    user_snapshots["transaction_count_7d"],
    color="#3b82f6", alpha=0.3, linewidth=1,
)

# Plot label events
for _, row in user_labels.iterrows():
    ax.axvline(row["label_timestamp"], color="#ef4444", linestyle="--", alpha=0.6)
    ax.annotate(
        f"Label event\n({'fraud' if row['is_fraud'] else 'legit'})",
        (row["label_timestamp"], ax.get_ylim()[1] * 0.9),
        fontsize=9, color="#ef4444", ha="center",
    )

# Plot the selected feature values (point-in-time result)
for _, row in user_training.iterrows():
    ax.scatter(
        row["feature_as_of_timestamp"],
        row["transaction_count_7d"],
        color="#22c55e", s=200, zorder=6, marker="*",
    )
    # Draw arrow from selected feature to label
    ax.annotate(
        "", xy=(row["label_timestamp"], row["transaction_count_7d"]),
        xytext=(row["feature_as_of_timestamp"], row["transaction_count_7d"]),
        arrowprops=dict(arrowstyle="->", color="#22c55e", lw=1.5),
    )

ax.set_xlabel("Time", fontsize=12)
ax.set_ylabel("transaction_count_7d", fontsize=12)
ax.set_title(
    f"Point-in-Time Join for {demo_user}\n"
    "Green stars = selected features (most recent BEFORE label event)",
    fontsize=13, fontweight="bold",
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Blue dots = feature snapshots (weekly updates)")
print("Red dashed = label events (when we need a prediction)")
print("Green stars = features used for training (most recent BEFORE label)")
print("\nNotice: we never use features from AFTER the label event.")

### 3.1 Demonstrate Data Leakage Impact

Let's compare model performance WITH and WITHOUT point-in-time correctness
to see how data leakage inflates metrics.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score

feature_cols = [
    "avg_transaction_amount_30d", "transaction_count_7d",
    "account_age_days", "is_verified", "risk_score", "device_count",
]

# --- Correct: Point-in-time features ---
correct_data = training_data.dropna(subset=feature_cols)
correct_data["is_verified"] = correct_data["is_verified"].astype(int)
X_correct = correct_data[feature_cols].values
y_correct = correct_data["is_fraud"].values

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_correct, y_correct, test_size=0.2, random_state=42, stratify=y_correct
)

model_correct = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model_correct.fit(X_train_c, y_train_c)
y_prob_correct = model_correct.predict_proba(X_test_c)[:, 1]

# --- Leaky: Use LATEST features regardless of timestamp ---
leaky_query = f"""
WITH latest_features AS (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY user_id ORDER BY feature_timestamp DESC
    ) AS rn
    FROM `{PROJECT_ID}.{BQ_DATASET}.user_features`
)
SELECT
    l.user_id, l.is_fraud,
    f.avg_transaction_amount_30d, f.transaction_count_7d,
    f.account_age_days, f.is_verified, f.risk_score, f.device_count
FROM `{PROJECT_ID}.{BQ_DATASET}.fraud_labels` l
JOIN latest_features f ON l.user_id = f.user_id AND f.rn = 1
"""

leaky_data = bq_client.query(leaky_query).to_dataframe()
leaky_data["is_verified"] = leaky_data["is_verified"].astype(int)
X_leaky = leaky_data[feature_cols].values
y_leaky = leaky_data["is_fraud"].values

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leaky, y_leaky, test_size=0.2, random_state=42, stratify=y_leaky
)

model_leaky = GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42)
model_leaky.fit(X_train_l, y_train_l)
y_prob_leaky = model_leaky.predict_proba(X_test_l)[:, 1]

# Compare
print("=== Impact of Point-in-Time Correctness ===")
print(f"\n{'Metric':<20} {'Correct (PIT)':>15} {'Leaky (Latest)':>15} {'Difference':>12}")
print("-" * 65)

for metric_name, metric_fn in [("AUC-ROC", roc_auc_score)]:
    correct_score = metric_fn(y_test_c, y_prob_correct)
    leaky_score = metric_fn(y_test_l, y_prob_leaky)
    diff = leaky_score - correct_score
    print(f"{metric_name:<20} {correct_score:>15.4f} {leaky_score:>15.4f} {diff:>+12.4f}")

for metric_name, metric_fn in [("Accuracy", accuracy_score), ("F1", f1_score)]:
    correct_score = metric_fn(y_test_c, (y_prob_correct > 0.5).astype(int))
    leaky_score = metric_fn(y_test_l, (y_prob_leaky > 0.5).astype(int))
    diff = leaky_score - correct_score
    print(f"{metric_name:<20} {correct_score:>15.4f} {leaky_score:>15.4f} {diff:>+12.4f}")

print("\nLeaky features often show inflated metrics that do not hold in production.")
print("Feature Store's point-in-time joins prevent this data leakage.")

---
## Section 4: Online Serving — Fetch Latest Feature Values

For real-time predictions, the online store provides **millisecond-latency lookups**
by entity ID. The prediction service fetches features just before calling the model.

### Architecture
1. Prediction request arrives with `user_id`
2. Serving code calls Feature Store online API: `fetch_feature_values(user_id)`
3. Feature Store returns latest feature values (from Bigtable)
4. Features are combined with request-level (inline) features
5. Combined features are sent to model endpoint for prediction

In [ ]:
# Create Feature Online Store and Feature View (production pattern)
# Note: Online store creation requires billing and takes a few minutes

# Step 1: Create an online store
ONLINE_STORE_NAME = "fraud-detection-online-store"

# Uncomment to create in production:
# online_store = feature_store.FeatureOnlineStore.create(
#     name=ONLINE_STORE_NAME,
#     feature_online_store_type=feature_store.utils.FeatureOnlineStoreType.BIGTABLE,
#     bigtable_auto_scaling=feature_store.utils.BigtableAutoScaling(
#         min_node_count=1,
#         max_node_count=3,
#         cpu_utilization_target=70,
#     ),
# )

# Step 2: Create a Feature View that selects features for this use case
# feature_view = feature_store.FeatureView.create(
#     name="fraud_detection_view",
#     feature_registry_source=feature_store.utils.FeatureViewFeatureRegistrySource(
#         feature_groups={
#             user_fg.resource_name: feature_store.utils.FeatureViewFeatureRegistrySource.FeatureGroup(
#                 features=[f.name for f in features],
#             ),
#         },
#     ),
# )

# Step 3: Sync to online store
# sync_job = online_store.create_feature_view_sync(feature_view=feature_view)

print("Online store pattern defined.")
print("In production, this creates a managed Bigtable instance for low-latency serving.")
print("\nOnline serving flow:")
print("  1. Prediction request: POST /predict {user_id: 'user_0042'}")
print("  2. Fetch features: online_store.fetch_feature_values('user_0042')")
print("  3. Response (ms): {avg_txn_amount_30d: 450.0, txn_count_7d: 12, ...}")
print("  4. Predict: endpoint.predict(features)")

In [ ]:
# Simulate online feature serving (what the API call looks like)
def simulate_online_serving(user_id: str, feature_data: pd.DataFrame) -> dict:
    """
    Simulate fetching latest feature values for a user.
    In production, this is a single API call to Feature Store.
    """
    user_data = feature_data[feature_data["user_id"] == user_id]
    if user_data.empty:
        return {"error": f"No features found for {user_id}"}

    # Get the most recent snapshot (online store always returns latest)
    user_data = user_data.copy()
    user_data["feature_timestamp"] = pd.to_datetime(user_data["feature_timestamp"])
    latest = user_data.sort_values("feature_timestamp").iloc[-1]

    return {
        "entity_id": user_id,
        "features": {
            "avg_transaction_amount_30d": float(latest["avg_transaction_amount_30d"]),
            "transaction_count_7d": int(latest["transaction_count_7d"]),
            "account_age_days": int(latest["account_age_days"]),
            "is_verified": bool(latest["is_verified"]),
            "risk_score": float(latest["risk_score"]),
            "device_count": int(latest["device_count"]),
        },
        "feature_timestamp": str(latest["feature_timestamp"]),
    }


# Simulate online lookups for several users
test_users = ["user_0001", "user_0042", "user_0100", "user_0150"]

print("=== Simulated Online Feature Serving ===")
print(f"{'User ID':<12} {'Avg Txn $':>10} {'Txn Count':>10} {'Age (days)':>10} {'Verified':>10} {'Risk':>8}")
print("-" * 68)

for uid in test_users:
    result = simulate_online_serving(uid, user_features_df)
    f = result["features"]
    print(
        f"{uid:<12} "
        f"${f['avg_transaction_amount_30d']:>8.2f} "
        f"{f['transaction_count_7d']:>10} "
        f"{f['account_age_days']:>10} "
        f"{str(f['is_verified']):>10} "
        f"{f['risk_score']:>8.4f}"
    )

print("\nIn production, each lookup takes <10ms via the Bigtable-backed online store.")

---
## Section 5: Feature Monitoring Configuration

Feature monitoring detects **drift** (distribution changes) and **staleness** (outdated features).
Both can degrade model performance silently.

### Key Monitoring Metrics
- **Jensen-Shannon divergence** (numerical features): measures distribution difference
- **L-infinity distance** (categorical features): max difference in category probabilities
- **Staleness**: time since last feature update

In [ ]:
from scipy import stats

def jensen_shannon_divergence(p: np.ndarray, q: np.ndarray, n_bins: int = 50) -> float:
    """Compute Jensen-Shannon divergence between two distributions."""
    # Create histograms with shared bins
    all_vals = np.concatenate([p, q])
    bins = np.linspace(all_vals.min(), all_vals.max(), n_bins + 1)

    p_hist, _ = np.histogram(p, bins=bins, density=True)
    q_hist, _ = np.histogram(q, bins=bins, density=True)

    # Add small epsilon to avoid log(0)
    p_hist = p_hist + 1e-10
    q_hist = q_hist + 1e-10

    # Normalize
    p_hist = p_hist / p_hist.sum()
    q_hist = q_hist / q_hist.sum()

    # JS divergence = average of KL divergences
    m = 0.5 * (p_hist + q_hist)
    jsd = 0.5 * (stats.entropy(p_hist, m) + stats.entropy(q_hist, m))
    return float(jsd)


# Simulate feature drift: compare training data vs "production" data
np.random.seed(42)

# Training distribution (original)
training_txn_amounts = training_data["avg_transaction_amount_30d"].dropna().values

# Production distribution (shifted -- simulating drift)
# Scenario: inflation causes transaction amounts to increase by 15%
production_txn_amounts = training_txn_amounts * 1.15 + np.random.normal(0, 20, len(training_txn_amounts))

jsd = jensen_shannon_divergence(training_txn_amounts, production_txn_amounts)
threshold = 0.3

print(f"Feature: avg_transaction_amount_30d")
print(f"Jensen-Shannon Divergence: {jsd:.4f}")
print(f"Threshold: {threshold}")
print(f"Status: {'DRIFT DETECTED -- trigger alert' if jsd > threshold else 'Within normal range'}")

In [ ]:
# Visualize feature drift across multiple features
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Simulate different levels of drift for each feature
drift_scenarios = [
    {"name": "avg_txn_amount_30d", "shift": 1.15, "noise": 20, "desc": "15% inflation"},
    {"name": "txn_count_7d", "shift": 1.0, "noise": 0.5, "desc": "No drift"},
    {"name": "account_age_days", "shift": 1.02, "noise": 5, "desc": "Slight aging"},
    {"name": "risk_score", "shift": 1.3, "noise": 0.05, "desc": "Risk increase"},
    {"name": "device_count", "shift": 1.5, "noise": 0.3, "desc": "More devices"},
    {"name": "is_verified", "shift": 1.0, "noise": 0, "desc": "Categorical"},
]

drift_results = []

for ax, scenario in zip(axes.flat, drift_scenarios):
    col_name = scenario["name"]

    if col_name in training_data.columns:
        train_vals = training_data[col_name].dropna().values.astype(float)
    else:
        train_vals = np.random.randn(200)

    prod_vals = train_vals * scenario["shift"] + np.random.normal(0, scenario["noise"], len(train_vals))

    if col_name != "is_verified":
        jsd = jensen_shannon_divergence(train_vals, prod_vals)
    else:
        jsd = 0.02  # Categorical feature, minimal drift

    drift_results.append({"feature": col_name, "jsd": jsd, "alert": jsd > 0.3})

    # Plot distributions
    ax.hist(train_vals, bins=30, alpha=0.5, color="#3b82f6", label="Training", density=True)
    ax.hist(prod_vals, bins=30, alpha=0.5, color="#ef4444", label="Production", density=True)

    drift_color = "#ef4444" if jsd > 0.3 else "#22c55e"
    ax.set_title(f"{col_name}\nJSD={jsd:.3f} ({scenario['desc']})", fontsize=10, color=drift_color)
    ax.legend(fontsize=8)
    ax.set_ylabel("Density", fontsize=9)

plt.suptitle(
    "Feature Drift Detection: Training vs Production Distributions",
    fontsize=14, fontweight="bold",
)
plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Feature':<30} {'JSD':>8} {'Status':<20}")
print("-" * 60)
for r in drift_results:
    status = "ALERT: DRIFT DETECTED" if r["alert"] else "OK"
    print(f"{r['feature']:<30} {r['jsd']:>8.4f} {status:<20}")

In [ ]:
# Feature monitoring configuration pattern (Vertex AI SDK)
# This is how you configure monitoring in production

monitoring_config = {
    "snapshot_analysis": {
        "monitoring_interval_days": 1,  # Check daily
    },
    "numerical_threshold_config": {
        "value": 0.3,  # Jensen-Shannon divergence threshold
    },
    "categorical_threshold_config": {
        "value": 0.3,  # L-infinity distance threshold
    },
}

# Staleness configuration
staleness_config = {
    "max_staleness_hours": 24,  # Alert if features not updated in 24h
    "notification_channels": [
        "projects/PROJECT/notificationChannels/CHANNEL_ID",
    ],
}

print("=== Feature Monitoring Configuration ===")
print(f"\nDrift Detection:")
print(f"  Check interval: {monitoring_config['snapshot_analysis']['monitoring_interval_days']} day(s)")
print(f"  Numerical threshold (JSD): {monitoring_config['numerical_threshold_config']['value']}")
print(f"  Categorical threshold (L-inf): {monitoring_config['categorical_threshold_config']['value']}")
print(f"\nStaleness Detection:")
print(f"  Max staleness: {staleness_config['max_staleness_hours']} hours")
print(f"\nWhen drift or staleness is detected:")
print(f"  1. Alert sent via Cloud Monitoring")
print(f"  2. Optional: trigger continuous training pipeline")
print(f"  3. Optional: trigger investigation workflow")

---
## Section 6: Feature Store vs Inline Features

Not all features belong in a Feature Store. Understanding the tradeoff is important
for the exam and for real-world system design.

In [ ]:
# Demonstrate the two feature patterns side by side

def predict_with_feature_store(user_id: str, request_data: dict) -> dict:
    """
    Pattern 1: Feature Store + inline features.

    Entity features (user history) come from Feature Store.
    Request features (transaction details) come from the request.
    """
    # Step 1: Fetch entity features from Feature Store (simulated)
    entity_features = simulate_online_serving(user_id, user_features_df)

    # Step 2: Combine with request-level (inline) features
    combined_features = {
        **entity_features["features"],     # From Feature Store
        "transaction_amount": request_data["amount"],           # Inline
        "transaction_hour": request_data["hour"],               # Inline
        "is_international": request_data["is_international"],   # Inline
    }

    return {
        "user_id": user_id,
        "feature_source": {
            "feature_store": list(entity_features["features"].keys()),
            "inline": ["transaction_amount", "transaction_hour", "is_international"],
        },
        "combined_features": combined_features,
    }


def predict_all_inline(user_id: str, request_data: dict) -> dict:
    """
    Pattern 2: All features inline (no Feature Store).

    All features must be computed at prediction time or sent with the request.
    This is simpler but prone to training-serving skew.
    """
    return {
        "user_id": user_id,
        "feature_source": {"inline": list(request_data.keys())},
        "features": request_data,
        "warning": "No Feature Store -- training-serving skew risk!",
    }


# Example prediction request
request = {
    "amount": 1500.00,
    "hour": 3,  # 3 AM (suspicious)
    "is_international": True,
}

# Pattern 1: Feature Store
result_fs = predict_with_feature_store("user_0042", request)
print("=== Pattern 1: Feature Store + Inline ===")
print(f"Feature Store features: {result_fs['feature_source']['feature_store']}")
print(f"Inline features:       {result_fs['feature_source']['inline']}")
print(f"Total features:        {len(result_fs['combined_features'])}")

# Pattern 2: All inline
result_inline = predict_all_inline("user_0042", request)
print(f"\n=== Pattern 2: All Inline ===")
print(f"Inline features: {result_inline['feature_source']['inline']}")
print(f"Warning: {result_inline.get('warning', 'None')}")

print("\n=== When to Use Each ===")
print("Feature Store: precomputed entity features (user history, aggregates)")
print("Inline:        request-level features (amount, time, device)")
print("Best practice: combine both patterns for production systems.")

In [ ]:
# Decision matrix visualization
decisions = [
    {"feature": "avg_txn_amount_30d", "store": True,
     "reasons": ["Expensive to compute", "Shared across models", "Requires history"]},
    {"feature": "transaction_count_7d", "store": True,
     "reasons": ["Aggregation over time window", "Needs point-in-time"]},
    {"feature": "account_age_days", "store": True,
     "reasons": ["Shared entity attribute", "Multiple teams use it"]},
    {"feature": "is_verified", "store": True,
     "reasons": ["Entity attribute", "Needs governance/auditing"]},
    {"feature": "transaction_amount", "store": False,
     "reasons": ["Available in request", "No history needed"]},
    {"feature": "transaction_hour", "store": False,
     "reasons": ["Derived from request timestamp", "Trivial computation"]},
    {"feature": "is_international", "store": False,
     "reasons": ["Available in request payload", "No entity lookup needed"]},
    {"feature": "merchant_category", "store": False,
     "reasons": ["Comes with transaction", "Request-level data"]},
]

print(f"{'Feature':<25} {'Where?':<18} {'Reasons'}")
print("=" * 80)
for d in decisions:
    where = "Feature Store" if d["store"] else "Inline"
    reasons = "; ".join(d["reasons"])
    print(f"{d['feature']:<25} {where:<18} {reasons}")

---
## Section 7: Cleanup

In [ ]:
# Cleanup: delete BigQuery dataset (optional)
# Uncomment to clean up resources:

# bq_client.delete_dataset(
#     f"{PROJECT_ID}.{BQ_DATASET}",
#     delete_contents=True,
#     not_found_ok=True,
# )
# print(f"Deleted BigQuery dataset: {BQ_DATASET}")

# Feature Store cleanup:
# user_fg.delete(force=True)  # Delete feature group and all features
# online_store.delete(force=True)  # Delete online store

print("Cleanup commands are commented out. Uncomment to delete resources.")
print("Remember to delete Feature Store resources to avoid ongoing charges.")

---
## Summary

In this notebook we covered the core Feature Store capabilities in Vertex AI:

1. **Feature Data in BigQuery** -- Created synthetic feature tables with timestamps, demonstrating the BigQuery-backed architecture where features are stored as standard BQ tables.

2. **Feature Groups and Registration** -- Created Feature Groups pointing to BigQuery tables and registered individual features with descriptions for discovery and governance.

3. **Point-in-Time Joins** -- Implemented batch feature serving with temporal correctness, demonstrated data leakage impact, and visualized how the join selects the correct historical features.

4. **Online Serving** -- Simulated low-latency entity lookups for real-time prediction, showing the API pattern for fetching latest feature values.

5. **Feature Monitoring** -- Implemented Jensen-Shannon divergence for drift detection, visualized training vs production distributions, and configured monitoring thresholds.

6. **Feature Store vs Inline** -- Compared both patterns side-by-side, built a decision matrix for when to use each, and demonstrated the combined pattern for production systems.

### Exam Takeaways
- **Online serving** = millisecond lookups for prediction; **Offline serving** = batch for training
- **Point-in-time correctness** prevents data leakage (only use features available at label time)
- **Feature Store** for shared, precomputed entity features; **Inline** for request-level features
- **Feature monitoring** detects drift (distribution changes) and staleness (outdated features)
- New architecture uses **Feature Groups + Feature Views**; legacy used **Featurestore + EntityTypes**

**Next notebook**: Course 11 covers Introduction to Generative AI on Google Cloud.